# Stock Market Prediction Using Machine Learning and Ensemble Learning

## Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import yfinance as yf
import matplotlib.pyplot as plt
import joblib

from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split,TimeSeriesSplit,KFold,cross_validate,RandomizedSearchCV)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error,r2_score)

sns.set_theme()

## Data Processing


In [ ]:
# Ticker = input(print('Enter Ticker: '))
Ticker ='MSFT'

df = yf.download(Ticker,'2020-01-01')

In [ ]:
df.head()

In [ ]:
df['Target'] = df['Close'].shift(-1)
df.head()

In [ ]:

df['MA_10'] = df['Close'].rolling(10).mean()
df['MA_50'] = df['Close'].rolling(50).mean()


df['Volatility'] = df['Close'].rolling(10).std()


df['Daily_Return'] = df['Close'].pct_change()

In [ ]:
df.dropna(axis=0,inplace=True)

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()


## Feature Selection

In [ ]:
target = df[['Target']]
features = df[['Close','Volume','High','Low','Open',
         'MA_10','MA_50',
         'Volatility']]

## Train Test Split

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(features,target,test_size = 0.2,shuffle = False)

## Standardization


In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit(x_train)

joblib.dump(scaler,"../model/scaler.pkl")


## TimeSeries Split 

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(tscv.split(x_train), 1):

    X_train_cv = x_train.iloc[train_idx]
    X_val_cv = x_train.iloc[val_idx]

    y_train_cv = y_train.iloc[train_idx]
    y_val_cv = y_train.iloc[val_idx]


## Model-1 Regression

In [ ]:
model = LinearRegression()

In [ ]:
model.fit(x_train,y_train)

In [ ]:
R_Squared = model.score(x_train, y_train)
Bias = model.intercept_[0]


summary = pd.DataFrame({
    'Feature': ['R-Squared', 'Bias (Intercept)'],
    'Value': [R_Squared, Bias]
})

summary

In [ ]:
y_pred = model.predict(x_test)

y_act = scaler_y.inverse_transform(y_pred)

y_test_act = scaler_y.inverse_transform(y_test)

## Model 2 Random Forest

In [ ]:
y_rf = df['Target']

## Train_Test

In [ ]:
x_train_rf, x_test_rf, y_train_rf, y_test_rf = train_test_split(x_scaled,y_rf,test_size=0.2,shuffle=False)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=800,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,
    random_state=42
)

rf.fit(x_train_rf,y_train_rf)

rf_pred = rf.predict(x_test_rf)

## Ensemble Learning

In [ ]:
ensemble_pred = (0.8 * y_pred +0.2 * rf_pred.reshape(-1,1))

##  Error Calculation

In [ ]:
comparison = pd.DataFrame({'Actual': y_test_act.flatten(),'Linear Regression': y_act.flatten(),'Random Forest': rf_pred.flatten()})

comparison['Ensemble'] = 0.8 * comparison['Linear Regression'] + 0.2 * comparison['Random Forest']

comparison['LR_Error'] = comparison['Actual'] - comparison['Linear Regression']
comparison['RF_Error'] = comparison['Actual'] - comparison['Random Forest']
comparison['Ensemble_Error'] = comparison['Actual'] - comparison['Ensemble']

comparison['LR_Smooth_Error'] = comparison['LR_Error'].rolling(10).mean()
comparison['RF_Smooth_Error'] = comparison['RF_Error'].rolling(10).mean()
comparison['Ensemble_Smooth_Error'] = comparison['Ensemble_Error'].rolling(10).mean()

In [ ]:
comparison['Date'] = df.index[-len(comparison):]

lr_rmse = np.sqrt(mean_squared_error(comparison['Actual'], comparison['Linear Regression']))
lr_mae = mean_absolute_error(comparison['Actual'], comparison['Linear Regression'])

rf_rmse = np.sqrt(mean_squared_error(comparison['Actual'], comparison['Random Forest']))
rf_mae = mean_absolute_error(comparison['Actual'], comparison['Random Forest'])

ensemble_rmse = np.sqrt(mean_squared_error(comparison['Actual'], comparison['Ensemble']))
ensemble_mae = mean_absolute_error(comparison['Actual'], comparison['Ensemble'])

In [ ]:
metrics = pd.DataFrame({'Model':['Linear Regression','Random Forest','Ensemble'],'RMSE':[lr_rmse,rf_rmse,ensemble_rmse],'MAE':[lr_mae,rf_mae,ensemble_mae]})

In [ ]:
metrics

In [ ]:
direction_acc = np.mean(np.sign(comparison['Actual'].diff()[1:]) == np.sign(comparison['Ensemble'].diff()[1:])) * 100

print(f'Direction Accuracy: {direction_acc:.2f}%')

## Chart

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(14,10),sharex=True,gridspec_kw={'height_ratios':[3,1]})

comparison.plot(x='Date',y=['Actual','Linear Regression','Random Forest','Ensemble'],ax=axes[0],linewidth=2,alpha=0.9,color=['red','cyan','orange','lime'],title=f'{Ticker} Stock Price Prediction')

axes[0].set_ylabel('Price')
axes[0].grid(alpha=0.3)


comparison.plot(x='Date',y=['LR_Smooth_Error','RF_Smooth_Error','Ensemble_Smooth_Error'],ax=axes[1],linewidth=2,title='Smoothed Prediction Error Graph')
axes[1].axhline(y=0,linestyle='--')
axes[1].set_ylabel('Error')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../images/Stock_analysis.png", dpi=300, bbox_inches='tight')
plt.show()